In [9]:
# =============================================================================
# CELL 1: CONFIGURATION, BASELINES, POOL DEFINITIONS
# =============================================================================

import pandas as pd

run_every_query = True

LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX']

BASELINES = {
    'AN':  {'ltv': 1.94, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
    'FRN': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25},
    'STG': {'ltv': 1.94, 'new_recovery_unadjusted': 0.60, 'apr': 0.25},
    'FLD': {'ltv': 1.45, 'new_recovery_unadjusted': 0.54, 'apr': 0.235},
    'ENT': {'ltv': 1.45, 'new_recovery_unadjusted': 0.55, 'apr': 0.235},
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
}

MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

# --- Load pools from CSV (row 1 has pool definitions, used as column headers automatically) ---
pool_csv = pd.read_csv('../queries/ragu_pool.csv', dtype=str)

POOLS = {}
for col in pool_csv.columns:
    accts = pool_csv[col].dropna().astype('int64').tolist()
    POOLS[col.strip()] = accts

all_pool_accounts = set()
for accts in POOLS.values():
    all_pool_accounts.update(accts)

for name, accts in POOLS.items():
    print(f'{name}: {len(accts)} accounts')
print(f'\nTotal unique accounts across all pools: {len(all_pool_accounts)}')

First group (8/1/2023 - 8/1/2024 loans above $80k):: 143 accounts
Second group (8/1/2024-8/1/2025 loans above $80k):: 228 accounts
Third group (2024 loans above $80k):: 169 accounts

Total unique accounts across all pools: 371


In [10]:
# =============================================================================
# CELL 2: IMPORTS AND UTILITY FUNCTIONS
# =============================================================================

import numpy as np
import pyodbc
import pickle
import warnings
import os
import openpyxl

pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)


def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag
    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag
    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)
    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)
    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)
    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag
    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag
    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date
    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag
    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag
    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag
    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)
    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag
    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)
    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)
    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))
    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))
    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)
    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))
    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag
    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))
    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)
    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))
    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age
    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc
    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag
    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag
    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag
    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag
    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag
    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag
    if leave_out != 'state_counter_adj':
        mask = ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag)
        ula_df.loc[mask, 'loss_multiplier'] *= 1.012
    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)
    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag
    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag
    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag
    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag
    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag
    if leave_out != 'blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1
    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)
    return ula_df


print('Utilities ready')

Utilities ready


In [11]:
# =============================================================================
# CELL 3: DATA FETCH (SQL + PICKLE)
# =============================================================================

os.makedirs('../../cache', exist_ok=True)
force = run_every_query

acct_str = ', '.join(str(a) for a in all_pool_accounts)
min_date_sql = f"'2000-01-01' AND cd.account_number IN ({acct_str})"

with pyodbc.connect("DSN=Redshift_prod_new") as conn:
    ula_df_total = cached_sql(
        '../queries/vintage_level_ula_query.txt', '../../cache/pool_ula.pkl',
        sub_list=[('{min_book_date}', min_date_sql)], connection=conn, force_refresh=force,
    )
    print(f'ULA ready: {len(ula_df_total):,} records')

    dla_df = cached_sql(
        '../queries/new_dll_query.txt', '../../cache/dla_v1.pkl',
        connection=conn, force_refresh=force,
    )
    print(f'DLA ready: {len(dla_df):,} records')

    new_recovery = cached_sql(
        '../queries/new_recovery_queryt.txt', '../../cache/pool_recovery.pkl',
        sub_list=[('{min_book_date}', min_date_sql)], connection=conn, force_refresh=force,
    )
    print(f'Recovery ready: {len(new_recovery):,} records')

ULA ready: 703 records
DLA ready: 134,149 records
Recovery ready: 1,217,667 records


In [12]:
# =============================================================================
# CELL 4: DATA PREP, FLAG CREATION, DLA MERGE
# =============================================================================

date_col = 'book_date'

ula_df_total = ula_df_total[ula_df_total.lob != 'Core']
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

date_col_str = f'{date_col}_str'
ula_df_total[date_col_str] = ula_df_total[date_col].astype(str)

# --- Flag creation ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- DLA Merge ---
dla_df = dla_df.rename(columns={'valid_vintage': 'book_vintage'})
dla_explicit = dla_df[dla_df['book_vintage'] != 'current']
dla_current = dla_df[dla_df['book_vintage'] == 'current'].drop(columns='book_vintage')
last_explicit_vintage = dla_explicit['book_vintage'].max()

ula_df_total = ula_df_total.merge(dla_explicit, how='left', on=['dealer_number', 'book_vintage'])
new_and_missing = ula_df_total['pricing_scalar'].isna() & (ula_df_total['book_vintage'] > last_explicit_vintage)
fallback = ula_df_total.loc[new_and_missing, ['dealer_number']].merge(dla_current, on='dealer_number', how='left')
for col in ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']:
    ula_df_total.loc[new_and_missing, col] = fallback[col].values

ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag ---
warnings.filterwarnings('ignore', category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings('default', category=UserWarning)

# --- NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- Filters (amt_financed cap removed since pools are explicitly >$80k) ---
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[
    (ula_df_total.lob == 'MCY') |
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]
print(f'ULA after filters: {len(ula_df_total):,}')

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- KMX Flags ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_toyho_flag'] = (ula_df_total.cd_model_score >= 130) & ula_df_total.make.isin({'HON', 'TOY'})
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- MTN 4.1 model score transformation ---
is_mtn41_ula = ula_df_total.mtn_model == 4.1
ula_df_total.loc[is_mtn41_ula, 'cd_model_score'] = (
    142 + (ula_df_total.loc[is_mtn41_ula, 'cd_model_score'] - 142) * 1.5
)

print(f'ULA after refinement: {len(ula_df_total):,}')
print(f'LOBs present: {sorted(ula_df_total.lob.unique())}')
print('[PROGRESS] Data Prep Complete')

ULA after filters: 687
ULA after refinement: 366
LOBs present: ['AN', 'FLD', 'FRN', 'KMX', 'STE', 'STG']
[PROGRESS] Data Prep Complete


In [13]:
# =============================================================================
# CELL 5: POOL-LEVEL RAGU SCORING
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def compute_pool_ragu(pool_accounts, ula_data, recovery_data):
    """Compute aggregate RAGU score for a pool of accounts, broken down by LOB."""
    ula_pool = ula_data[ula_data.account_number.isin(pool_accounts)].copy()

    if len(ula_pool) == 0:
        return None, 0

    lob_results = []

    for lob in LOBS:
        ula_lob = ula_pool[ula_pool.lob == lob].copy()
        if len(ula_lob) == 0:
            continue

        baseline_config = BASELINES[lob]
        baseline_ltv = baseline_config['ltv']
        baseline_recovery = baseline_config['new_recovery_unadjusted']
        baseline_apr = baseline_config['apr']
        ltv_mult = 17 / 0.65 if lob == 'KMX' else 17
        apr_mult = 0.7 / 0.65 if lob == 'KMX' else 0.7

        if lob == 'KMX':
            ula_lob = get_ula_multiplier_kmx(ula_lob)
        else:
            ula_lob = get_ula_multiplier_nonkmx(ula_lob)

        ula_lob = ula_lob[['account_number', date_col, 'bbvalue', 'sale_price',
                           'amt_financed', 'lob_or_bucket', 'lob',
                           'loss_multiplier', 'apr', 'cd_model_score']]

        nr = recovery_data[['account_number', 'new_recovery_multiplier']].drop_duplicates(
            subset='account_number', keep='first')
        mix_df = ula_lob.merge(
            nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
            on='account_number', how='left'
        ).drop_duplicates(subset='account_number', keep='first')

        mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

        bb_populated = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
        if len(bb_populated) == 0:
            continue

        total_af = bb_populated.amt_financed.sum()
        w_loss_mult = (bb_populated.loss_multiplier * bb_populated.amt_financed).sum() / total_af
        w_ltv = (bb_populated.ltv * bb_populated.amt_financed).sum() / total_af
        w_apr = (bb_populated.apr * bb_populated.amt_financed).sum() / total_af
        w_model_score = (bb_populated.cd_model_score * bb_populated.amt_financed).sum() / total_af

        recovery_pop = bb_populated[bb_populated['recovery_multiplier'].notna()]
        if len(recovery_pop) > 0:
            w_recovery = (recovery_pop.recovery_multiplier * recovery_pop.amt_financed).sum() / recovery_pop.amt_financed.sum()
        else:
            w_recovery = baseline_recovery

        unit_loss_score = w_model_score + (1 - w_loss_mult) * mean_unit_loss / unit_loss_to_model_score
        baselined_recovery = w_recovery / baseline_recovery

        gross_loss_impact = unit_loss_score - w_model_score
        recovery_impact = unit_loss_score * mean_unit_loss * w_recovery * (baselined_recovery - 1)
        ltv_impact = (baseline_ltv / w_ltv - 1) * ltv_mult
        apr_impact = (baseline_apr - w_apr) / 0.01 * apr_mult
        ragu_score = (
            (1 - mean_unit_loss * w_recovery) * unit_loss_score
            + mean_unit_loss * w_recovery * unit_loss_score * baselined_recovery
            + ltv_impact + apr_impact
        )

        lob_results.append({
            'lob': lob,
            'ms_original': w_model_score,
            'gross_loss_impact': gross_loss_impact,
            'recovery_impact': recovery_impact,
            'ltv_impact': ltv_impact,
            'apr_impact': apr_impact,
            'ragu_score': ragu_score,
            'amt_financed': total_af,
            'ltv': w_ltv,
            'apr': w_apr,
            'n_loans': len(bb_populated),
        })

    if not lob_results:
        return None, len(ula_pool)

    lob_df = pd.DataFrame(lob_results)

    # Rolled-up combined score across LOBs
    rollup_metrics = ['ms_original', 'gross_loss_impact', 'recovery_impact',
                      'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr']
    total_af = lob_df.amt_financed.sum()
    combined = {'lob': 'Combined', 'amt_financed': total_af, 'n_loans': lob_df.n_loans.sum()}
    for metric in rollup_metrics:
        combined[metric] = (lob_df[metric] * lob_df.amt_financed).sum() / total_af
    lob_df = pd.concat([lob_df, pd.DataFrame([combined])], ignore_index=True)

    return lob_df, len(ula_pool)


# --- Score each pool ---
pool_results = {}

for pool_name, pool_accounts in POOLS.items():
    result_df, n_matched = compute_pool_ragu(set(pool_accounts), ula_df_total, new_recovery)
    pool_results[pool_name] = result_df

    if result_df is not None:
        combined_row = result_df[result_df.lob == 'Combined'].iloc[0]
        n_lobs = len(result_df) - 1
        print(f'{pool_name}:')
        print(f'  Matched loans: {n_matched}, LOBs: {n_lobs}, RAGU Score: {combined_row.ragu_score:.4f}')
    else:
        print(f'{pool_name}: no scoreable loans found')

print('\n[PROGRESS] Pool RAGU Scoring Complete')

First group (8/1/2023 - 8/1/2024 loans above $80k)::
  Matched loans: 140, LOBs: 5, RAGU Score: 160.2985
Second group (8/1/2024-8/1/2025 loans above $80k)::
  Matched loans: 226, LOBs: 5, RAGU Score: 158.5838
Third group (2024 loans above $80k)::
  Matched loans: 166, LOBs: 5, RAGU Score: 161.7252

[PROGRESS] Pool RAGU Scoring Complete


In [14]:
# =============================================================================
# CELL 6: EXCEL EXPORT + DISPLAY
# =============================================================================

EXCEL_OUTPUT = '../output/pool_ragu.xlsx'
os.makedirs('../output', exist_ok=True)

METRIC_ROWS = [
    ('Model Score',                'ms_original'),
    ('Expected Gross Loss Impact', 'gross_loss_impact'),
    ('Recovery Impact',            'recovery_impact'),
    ('LTV Impact',                 'ltv_impact'),
    ('APR Impact',                 'apr_impact'),
    ('RAGU Score',                 'ragu_score'),
    ('Amount Financed',            'amt_financed'),
    ('Weighted LTV',               'ltv'),
    ('Weighted APR',               'apr'),
    ('Loan Count',                 'n_loans'),
]

wb = openpyxl.Workbook()
ws = wb.active
ws.title = 'Pool RAGU Scores'

current_row = 1

for pool_name, result_df in pool_results.items():
    if result_df is None:
        continue

    ws.cell(row=current_row, column=1, value=pool_name)
    current_row += 1

    lob_list = result_df.lob.tolist()
    ws.cell(row=current_row, column=1, value='Metric')
    for col_idx, lob in enumerate(lob_list, start=2):
        ws.cell(row=current_row, column=col_idx, value=lob)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws.cell(row=current_row, column=1, value=label)
        for col_idx, lob in enumerate(lob_list, start=2):
            row_data = result_df[result_df.lob == lob]
            if not row_data.empty:
                ws.cell(row=current_row, column=col_idx, value=row_data.iloc[0][col_key])
        current_row += 1

    current_row += 2

wb.save(EXCEL_OUTPUT)
print(f'Saved to {EXCEL_OUTPUT}')

# --- Display results inline ---
for pool_name, result_df in pool_results.items():
    if result_df is not None:
        print(f'\n=== {pool_name} ===')
        display(result_df.set_index('lob')[['ms_original', 'gross_loss_impact', 'recovery_impact',
                                            'ltv_impact', 'apr_impact', 'ragu_score',
                                            'amt_financed', 'ltv', 'apr', 'n_loans']].round(4))

Saved to ../output/pool_ragu.xlsx

=== First group (8/1/2023 - 8/1/2024 loans above $80k): ===


,ms_original,gross_loss_impact,recovery_impact,ltv_impact,apr_impact,ragu_score,amt_financed,ltv,apr,n_loans
lob,,,,,,,,,,
AN,141.3282,-2.8740,9.2642,15.4305,-0.3100,162.8389,420109.78,1.0169,0.2544,4
FRN,146.3292,0.5107,4.4400,12.9708,1.9377,166.1884,2971715.90,1.1004,0.2223,30
STG,142.3683,2.5999,3.7191,12.0159,1.4427,162.1458,2436747.80,1.1366,0.2294,27
FLD,144.7267,-8.0713,1.0926,7.3818,3.4847,148.6146,2352317.18,1.0110,0.1852,21
KMX,157.1340,5.5023,3.1425,12.1948,-0.3613,177.6122,254864.01,1.0844,0.2534,3
Combined,144.8156,-1.2967,3.4994,11.2355,2.0447,160.2985,8435754.67,1.0813,0.2166,85



=== Second group (8/1/2024-8/1/2025 loans above $80k): ===


,ms_original,gross_loss_impact,recovery_impact,ltv_impact,apr_impact,ragu_score,amt_financed,ltv,apr,n_loans
lob,,,,,,,,,,
AN,141.0000,-6.1607,-4.4465,8.1598,-2.0930,136.4596,81860.74,1.3108,0.2799,1
FRN,146.4762,0.8720,2.2521,14.2938,0.2384,164.1325,3413903.08,1.0539,0.2466,38
STG,142.7426,1.4011,1.4875,12.1555,1.0406,158.8273,2169310.51,1.1312,0.2351,24
FLD,145.2242,2.0719,-2.0983,6.8939,1.0827,153.1745,1349193.68,1.0316,0.2195,15
KMX,144.7973,2.8367,-1.2530,-3.1007,-1.6851,141.5952,609931.52,1.8039,0.2656,7
Combined,144.9992,1.3165,0.9124,10.9185,0.4371,158.5838,7624199.53,1.1347,0.2404,85



=== Third group (2024 loans above $80k): ===


,ms_original,gross_loss_impact,recovery_impact,ltv_impact,apr_impact,ragu_score,amt_financed,ltv,apr,n_loans
lob,,,,,,,,,,
AN,141.4405,-2.0398,6.9193,17.1304,-2.0930,161.3574,312993.34,0.9663,0.2799,3
FRN,147.3445,0.0624,4.4476,12.5941,1.8484,166.2969,3351568.63,1.1144,0.2236,34
STG,141.6015,3.2807,3.5987,11.8551,0.9407,161.2769,2129207.38,1.1430,0.2366,23
FLD,146.2752,-5.1304,-0.2368,8.4448,3.0501,152.4030,1309209.90,0.9688,0.1914,13
KMX,151.6284,4.2282,-1.5182,4.6593,-3.2308,155.7669,343756.68,1.3496,0.2800,4
Combined,145.4640,0.1736,3.2098,11.4777,1.4000,161.7252,7446735.93,1.1016,0.2266,77
